In [2]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix, roc_auc_score, precision_score, f1_score, recall_score, accuracy_score
)

In [3]:
import joblib
import numpy as np

preprocessor = joblib.load("../artifacts/preprocessor.pkl")
X_train = np.load("../artifacts/X_train.npy")
X_test = np.load("../artifacts/X_test.npy")
y_train = np.load("../artifacts/y_train.npy")
y_test = np.load("../artifacts/y_test.npy")

In [4]:
# 1. Point to the database one folder up
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

experiment_name = "loan_default_prediction"

# 2. Check if the experiment exists. If it doesn't, create it with the custom artifact location.
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(
        name=experiment_name, 
        artifact_location="file:../mlruns"
    )

# 3. Set the experiment as the active one
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location=('file:d:/@menuka/@Learn/machine '
 'learning/@Projects/loan-default-predictor/notebooks/../mlruns'), creation_time=1785242147665, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785242147665, lifecycle_stage='active', name='loan_default_prediction', tags={}, trace_location=None, workspace='default'>

In [5]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    
    "XGboost": XGBClassifier(
        n_estimators=100,
        learning_rate = 0.1,
        max_depth = 6,
        random_state=42,
        eval_metric="logloss"
    )
}

In [7]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("y_train unique values:", np.unique(y_train))
print("y_test unique values:", np.unique(y_test))
print("y_train first 5:", y_train[:5])
print("X_train first row first 5 values:", X_train[0, :5])
print("Any NaN in X_train:", np.isnan(X_train).any())
print("Any NaN in X_test:", np.isnan(X_test).any())

X_train shape: (118936, 46)
X_test shape: (29734, 46)
y_train shape: (118936,)
y_test shape: (29734,)
y_train unique values: [0 1]
y_test unique values: [0 1]
y_train first 5: [0 0 1 0 0]
X_train first row first 5 values: [ 0.         -0.67965894 -2.36445142 -1.9544583  -0.80986148]
Any NaN in X_train: False
Any NaN in X_test: False


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print("Unique predictions:", np.unique(y_pred))
print("Prediction distribution:", np.bincount(y_pred))
print("Prob range:", y_prob.min(), y_prob.max())
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("First 10 probs:", y_prob[:10])

Unique predictions: [0 1]
Prediction distribution: [22406  7328]
Prob range: 6.098473740373746e-07 0.9999999184514952
ROC AUC: 1.0
First 10 probs: [3.46884625e-05 1.52632304e-04 5.87880276e-06 2.08814661e-05
 1.12290020e-05 1.93394230e-05 4.69179480e-05 4.19816611e-05
 3.42060449e-05 2.77460364e-05]


In [6]:
for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):    
        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
    
        roc_auc = roc_auc_score(y_test, y_prob)
        
        
        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)

        # Log model
        if model_name == "XGboost":
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, name="model")

        print(f"{model_name}")
        print(f"Accuracy : {accuracy:.4f}")
        print(f"ROC AUC  : {roc_auc:.4f}")
        print("-" * 40)
    

LogisticRegression
Accuracy : 1.0000
ROC AUC  : 1.0000
----------------------------------------
RandomForestClassifier
Accuracy : 1.0000
ROC AUC  : 1.0000
----------------------------------------


2026/07/28 18:44:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGboost
Accuracy : 1.0000
ROC AUC  : 1.0000
----------------------------------------
